# RMSNorm 讲解与实现

这个 notebook 解释 `RMSNorm`（Root Mean Square Normalization）的核心思想，并给出一个简单的 PyTorch 实现。

它在很多现代大模型里很常见，因为它比 `LayerNorm` 更简单，通常也更稳定高效。

## RMSNorm 的直觉

`LayerNorm` 会先减去均值，再除以标准差；而 `RMSNorm` 只根据均方根做缩放，不做中心化。

如果输入向量是 `x`，那么 RMSNorm 可以写成：

$$
\text{RMS}(x) = \sqrt{\frac{1}{d}\sum_{i=1}^{d} x_i^2 + \epsilon}
$$

$$
\text{RMSNorm}(x) = \frac{x}{\text{RMS}(x)} \odot \gamma
$$

其中：

- `d` 是最后一维长度
- `epsilon` 用来防止除零
- `gamma` 是可学习参数，用于重新缩放各个通道

In [1]:
import torch
import torch.nn as nn

torch.manual_seed(7)
torch.set_printoptions(precision=4, sci_mode=False)

## 从零实现 RMSNorm

下面这个实现只针对最后一维做归一化，这也是 Transformer 中最常见的用法。

关键步骤：

1. 计算每个 token 在最后一维上的平方均值
2. 加上 `eps` 后开根号，得到 RMS
3. 用原向量除以 RMS
4. 乘上可学习参数 `weight`

和 LayerNorm 相比，这里没有减均值。

In [2]:
class RMSNorm(nn.Module):
    def __init__(self, dim: int, eps: float = 1e-8):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        rms = torch.sqrt(torch.mean(x * x, dim=-1, keepdim=True) + self.eps)
        x_norm = x / rms
        return x_norm * self.weight

## 在一个张量上试运行

这里构造一个三维输入 `(batch, seq_len, hidden_dim)`，模拟 Transformer block 中的常见输入形状。

In [3]:
batch_size = 2
seq_len = 3
hidden_dim = 6

x = torch.randn(batch_size, seq_len, hidden_dim)
rmsnorm = RMSNorm(hidden_dim)
y = rmsnorm(x)

print("input shape :", x.shape)
print("output shape:", y.shape)
print()
print("input[0, 0] =", x[0, 0])
print("output[0, 0] =", y[0, 0])

input shape : torch.Size([2, 3, 6])
output shape: torch.Size([2, 3, 6])

input[0, 0] = tensor([-0.8201,  0.3956,  0.8989, -1.3884, -0.1670,  0.2851])
output[0, 0] = tensor([-1.0481,  0.5056,  1.1487, -1.7743, -0.2134,  0.3644],
       grad_fn=<SelectBackward0>)


## 验证归一化后的 RMS

如果 `weight` 还全是 1，那么输出向量的均方根应该接近 1。我们来检查一下。

In [4]:
rms_after = torch.sqrt(torch.mean(y * y, dim=-1))
print(rms_after)

tensor([[1.0000, 1.0000, 1.0000],
        [1.0000, 1.0000, 1.0000]], grad_fn=<SqrtBackward0>)


## RMSNorm 和 LayerNorm 的区别

下面我们把两者放在一起对比：

- `LayerNorm`：会减均值，再按方差缩放
- `RMSNorm`：不减均值，只按 RMS 缩放

这意味着 RMSNorm 保留了原始表示的“均值信息”，同时减少了一些计算。

In [5]:
layernorm = nn.LayerNorm(hidden_dim)
ln_out = layernorm(x)

print("LayerNorm output[0, 0] =", ln_out[0, 0])
print("RMSNorm  output[0, 0] =", y[0, 0])
print()
print("LayerNorm mean of token:", ln_out[0, 0].mean())
print("RMSNorm mean of token :", y[0, 0].mean())

LayerNorm output[0, 0] = tensor([-0.8915,  0.6850,  1.3376, -1.6283, -0.0445,  0.5417],
       grad_fn=<SelectBackward0>)
RMSNorm  output[0, 0] = tensor([-1.0481,  0.5056,  1.1487, -1.7743, -0.2134,  0.3644],
       grad_fn=<SelectBackward0>)

LayerNorm mean of token: tensor(0.0000, grad_fn=<MeanBackward0>)
RMSNorm mean of token : tensor(-0.1695, grad_fn=<MeanBackward0>)


## 在大模型里的意义

RMSNorm 常用于 LLM，原因通常有几个：

- 实现更简单
- 省掉中心化步骤，计算更轻
- 在大规模训练里表现稳定
- 很适合配合 Pre-Norm Transformer 结构

像 LLaMA 一类模型中，就常能看到 RMSNorm 的使用。

## 小结

可以把 RMSNorm 理解为：

- 只管“尺度”，不管“中心”
- 保持实现简洁，同时保留较好的训练稳定性

如果你想继续练习，下一步可以试着补上：

- bias 版本的实现
- 和官方 / 第三方实现做数值对齐
- 把 RMSNorm 集成进一个简化版 Transformer block